# Protenix Structure Prediction

This notebook demonstrates multi-modal structure prediction using Protenix, an open-source reimplementation of AlphaFold3 developed by ByteDance Research. Protenix predicts the 3D structures of proteins, nucleic acids (DNA and RNA), small-molecule ligands, and their complexes. The example here predicts a protein-ligand complex: the MfnG protein bound to L-tyrosine. It covers input construction, model configuration, result inspection, visualization, and export.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("protenix")
display_overview("protenix")
display_docs_section("protenix", "Background")

# Protenix

Protenix is [ByteDance](https://www.bytedance.com/)'s open-source reproduction of AlphaFold3: a trainable PyTorch model that predicts the joint 3D structure of complexes mixing proteins, DNA, RNA, and small-molecule ligands, including modified residues. This toolkit runs Protenix structure prediction, with optional ColabFold multiple-sequence alignments and a choice of the base, mini, and tiny model variants that trade accuracy for speed.

Protenix ([ByteDance Research, 2025](https://doi.org/10.1101/2025.01.08.631967)) predicts the joint 3D structure of a biomolecular assembly from the sequences and chemical components it contains. It is a trainable, openly licensed reproduction of the AlphaFold3 architecture: like AlphaFold3, one model folds complexes that mix proteins, DNA, RNA, and small-molecule ligands, and predicts how those components are arranged relative to one another. Each protein chain can be paired with a multiple-sequence alignment (MSA) of evolutionarily related sequences, whose covariation patterns supply the evolutionary signal the model uses to place residues.

Architecturally, Protenix follows AlphaFold3 rather than AlphaFold2: it carries a single representation of the input tokens and a pairwise representation over token pairs, refines them through a Pairformer trunk, and generates all-atom coordinates with a diffusion module that starts from noise and iteratively denoises into a structure, in place of AlphaFold2's structure module. Several structures are sampled per random seed and ranked by a confidence score. Protenix is distributed in several sizes: full-parameter base models for highest accuracy, and lighter mini and tiny variants for faster, lower-memory prediction; the `mini_esm` and `mini_ism` variants replace the MSA with learned embeddings — from the ESM-2 protein language model or the ISM inverse-structure model, respectively — so they can fold without an alignment. Predicted confidence includes a per-residue predicted local distance difference test (pLDDT) for local reliability, a predicted aligned error (PAE) for the relative placement of any two tokens, a global predicted distance error (gPDE), and predicted template-modeling (pTM) and interface predicted template-modeling (ipTM) scores that summarize overall and interface accuracy.

The reference implementation is open-sourced at [bytedance/Protenix](https://github.com/bytedance/Protenix), with both the code and the model parameters released under the Apache-2.0 license for academic and commercial use. It was developed by [ByteDance](https://www.bytedance.com/)'s AI4Science team as a comprehensive reproduction of AlphaFold3, trained on comparable data to reach competitive accuracy across protein, nucleic-acid, and protein-ligand benchmarks.

## Available tools

In [2]:
display_available_tools("protenix")

- **`run_protenix()`** — Multi-modal structure prediction using Protenix (open-source AlphaFold3)

### `run_protenix`

Protenix is ByteDance Research's open-source reimplementation of AlphaFold3. It supports protein, DNA, RNA, and ligand inputs in a single unified model. For each complex, Protenix runs multiple diffusion samples and returns the highest-confidence prediction according to a ranking score that combines pTM, ipTM, pLDDT, and global predicted distance error. The base model with MSA integration delivers production-quality accuracy comparable to the original AlphaFold3, while mini and tiny variants offer faster inference for screening workflows.

In [3]:
from pathlib import Path

from proto_tools import (
    Chain,
    ProtenixConfig,
    ProtenixInput,
    Complex,
    run_protenix,
)

In [4]:
# Display input API reference
display_api_reference("protenix", "input", "run_protenix")

# MfnG protein sequence (384 residues)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = Complex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = ProtenixInput(complexes=[complex])

**Input** — `ProtenixInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `list[proto_tools.entities.complex.Complex]` | required | List of complexes to predict structure for containing chains and entity types. |
| `msas` | `dict[str, proto_tools.entities.msa.MSA] | None` | `None` | Pre-computed MSAs keyed by sequence. Populated by preprocess() or supplied directly. |

In [5]:
# Display config API reference
display_api_reference("protenix", "config", "run_protenix")

# Configure Protenix using the base model with MSA integration
config = ProtenixConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    model_name="protenix_base_default_v1.0.0",
    use_msa=True,
    num_pairformer_cycles=10,
    num_diffusion_samples=5,
    num_diffusion_steps=200,
    seeds=[0],
)

**Config** — `ProtenixConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on (e.g., 'cuda', 'cpu') |
| `timeout` | `int | None` | `1200` | Maximum execution time in seconds (base models typically need 10-15 min on slower GPUs). |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `include_pae_matrix` | `bool` | `False` | Attach the full per-residue PAE matrix. |
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains using ColabFold search |
| `colabfold_search_config` | `proto_tools.tools.sequence_alignment.colabfold_search.colabfold_search.ColabfoldSearchConfig | None` | `None` | Nested configuration for ColabFold MSA search. If None, uses default settings. |
| `model_name` | `Literal['protenix_base_default_v1.0.0', 'protenix_base_20250630_v1.0.0', 'protenix_base_default_v0.5.0', 'protenix_base_constraint_v0.5.0', 'protenix_mini_esm_v0.5.0', 'protenix_mini_ism_v0.5.0', 'protenix_mini_default_v0.5.0', 'protenix_tiny_default_v0.5.0', 'protenix-v2']` | `'protenix_base_default_v1.0.0'` | Protenix model variant to use for structure prediction |
| `seeds` | `list[int]` | `[0]` | Random seeds for sampling; each seed produces num_diffusion_samples independent samples. |
| `num_diffusion_samples` | `int` | `5` | Structure samples per seed; best by ranking score is kept. Higher = more thorough but slower. |
| `num_diffusion_steps` | `int | None` | `None` | Denoising steps. Default depends on model_name (base/constraint=200, mini/tiny=5). |
| `num_pairformer_cycles` | `int | None` | `None` | Pairformer refinement passes. Default depends on model_name (base/constraint=10, mini/tiny=4). |

In [6]:
# Run the prediction
result = run_protenix(inputs, config)

Running run_colabfold_search [00:00]

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [7]:
# Display output API reference
display_api_reference("protenix", "output", "run_protenix")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"Number of chains:  {len(complex.chains)}")
print(f"Protein length:    {len(mfng_sequence)} residues")
print(f"Confidence score:  {mfng_structure.metrics.confidence_score:.3f}")
print(f"Average pLDDT:     {mfng_structure.metrics.avg_plddt:.3f}")
print(f"pTM score:         {mfng_structure.metrics.ptm:.3f}")
print(f"ipTM score:        {mfng_structure.metrics.iptm:.3f}")
print(f"GPDE:              {mfng_structure.metrics.get('gpde', 'N/A')}")

# Visualize predicted complex colored by pLDDT confidence
mfng_structure.visualize(style="cartoon", color_by="bfactor")

**Output** — `ProtenixOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `list[proto_tools.entities.structures.structure.Structure]` | required | List of predicted structures |

Number of chains:  2
Protein length:    384 residues
Confidence score:  0.866
Average pLDDT:     0.890
pTM score:         0.898
ipTM score:        0.858
GPDE:              0.41768020391464233


## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD, and as input to molecular dynamics or further design pipelines. The B-factor column contains pLDDT confidence scores for per-residue visualization in any compatible viewer.

In [8]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")